In [2]:
from datasets import load_dataset
import re
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Alonso\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [3]:
repo_id = "astroza/chilean-humor-raw-transcripts"
ds = load_dataset(repo_id, "segments", split="train")
segments = ds["text"]

ONLY_NUMERIC_RE = re.compile(r"^[\s0-9,.;:/+\-()\[\]{}]+$")
LETTER_RE = re.compile(r"[A-Za-zÁÉÍÓÚáéíóúÑñÜü]")

def is_numeric_noise(text: str) -> bool:
    if not text:
        return False
    t = text.strip()
    if not t:
        return False
    if LETTER_RE.search(t):
        return False
    return bool(ONLY_NUMERIC_RE.fullmatch(t))

new_ds = ds.filter(lambda x: not is_numeric_noise(x["text"]))
segments = new_ds["text"]

print(f"Total: {len(segments)} segments")

README.md: 0.00B [00:00, ?B/s]

segments/train-00000-of-00001.parquet:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4645 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4645 [00:00<?, ? examples/s]

Total: 4591 segments


In [4]:

spanish_stopwords = stopwords.words("spanish")

vectorizer_model = CountVectorizer(stop_words=spanish_stopwords,
                                   min_df=2,
                                   ngram_range=(1, 2),
                                   token_pattern = r"(?ui)(?:^|[^\wáéíóúñü])([a-záéíóúñü][\wáéíóúñü]+)")

In [5]:
topic_model = BERTopic(language="multilingual",
                       vectorizer_model=vectorizer_model,
                       calculate_probabilities=True,
                       verbose=True)

topics, probs = topic_model.fit_transform(segments)

2026-02-10 10:15:46,334 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/144 [00:00<?, ?it/s]

2026-02-10 10:17:16,000 - BERTopic - Embedding - Completed ✓
2026-02-10 10:17:16,002 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-10 10:17:35,635 - BERTopic - Dimensionality - Completed ✓
2026-02-10 10:17:35,637 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-10 10:17:36,257 - BERTopic - Cluster - Completed ✓
2026-02-10 10:17:36,267 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-10 10:17:37,199 - BERTopic - Representation - Completed ✓


In [6]:
freq = topic_model.get_topic_info(); freq.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2590,-1_ahí_huevón_si_así,"[ahí, huevón, si, así, gracias, ah, cómo, voy,...","[Muchas gracias. Estaba enojada, mi marido me ..."
1,0,149,0_mamá_papá_hija_hijo,"[mamá, papá, hija, hijo, hijos, voy, casa, año...","[que sé que hoy día es una noche de jóvenes, p..."
2,1,148,1_humor_quinta_vergara_quinta vergara,"[humor, quinta, vergara, quinta vergara, escen...","[Muchas gracias, viña. Un día sin reír es un d..."
3,2,111,2_doctor_dijo_dice_señora,"[doctor, dijo, dice, señora, farmacia, médico,...","[Doctor le dice, ""Mire, su señora veo que le f..."
4,3,90,3_canción_mosca_mosca mosca_música,"[canción, mosca, mosca mosca, música, canción ...","[Con una canción maravillosa que dice, ""Las em..."
5,4,87,4_dijo_dice_dije_señora,"[dijo, dice, dije, señora, oye, ahí, si, ah, a...","[Oye, el tipo está en el restaurante. Él dice,..."
6,5,85,5_presidente_huevón_país_ser,"[presidente, huevón, país, ser, políticos, pol...","[Pero es una maravilla. Ahora, el 72 me pasó i..."
7,6,79,6_huevón_huevada_ahí_gallo,"[huevón, huevada, ahí, gallo, huevona, vamos, ...","[en el en el bosque, huevón, y en vez de manda..."
8,7,76,7_almas_amor_amas_vida,"[almas, amor, amas, vida, dio, quiero, dime, a...",[Y todo me dio el amor. Y todo me dio el amor....
9,8,72,8_quinta_vergara_quinta vergara_gracias,"[quinta, vergara, quinta vergara, gracias, muc...","[Que sigo siendo aquel, el mismo. Muchas graci..."


In [7]:
fig = topic_model.visualize_topics(); fig

In [10]:
dates = new_ds["date"]


decades = [(int(d[:4]) // 10) * 10 for d in dates]

topics_over_time = topic_model.topics_over_time(
    docs=segments,
    timestamps=decades,
    global_tuning=True,
    evolution_tuning=True,
)

6it [00:01,  3.58it/s]


In [11]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=20)

In [12]:
topic_model.visualize_topics_over_time(topics_over_time, topics=[11, 12, 13, 18])